# 🔄 Self-RAG — The LLM Checks Its Own Work

## The Idea

Normal RAG always retrieves, always uses what it found, always generates.

**Self-RAG** adds self-reflection at each step:

```
Question
  ↓
Do I even need to retrieve?  →  No → Just answer directly
  ↓ Yes
Retrieve docs
  ↓
Are these docs relevant?  →  No → Retrieve again
  ↓ Yes
Generate answer
  ↓
Is my answer supported by the docs?  →  No → Regenerate
  ↓ Yes
Return answer
```

## Why This Matters

- Sometimes you **don't need retrieval** ("What is 2+2?")
- Sometimes retrieved docs are **irrelevant** — better to try again than use them
- Sometimes the **answer contradicts the docs** — catch it before the user sees it

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

docs = [
    "Python lists are ordered and allow duplicate values.",
    "Python dictionaries store key-value pairs and are unordered in Python 3.6 and earlier.",
    "The append() method adds an item to the end of a list.",
    "Sets in Python store unique values only.",
]

doc_embeddings = model.encode(docs)
print("Ready!")

In [ ]:
# The three self-reflection checks

# Check 1: Does this question need retrieval at all?
# Simple heuristic: factual/math questions don't need retrieval
NO_RETRIEVAL_KEYWORDS = ["what is", "calculate", "how much is", "what's", "define"]

def needs_retrieval(question):
    q = question.lower()
    # If question is about simple facts or math, skip retrieval
    if any(q.startswith(kw) for kw in ["what is 2", "calculate", "how much is"]):
        return False
    return True  # Default: retrieve

test_questions = [
    "What is 2 + 2?",
    "How do Python lists work?",
    "Calculate 100 / 4",
    "What is the append method?",
]

print("Should we retrieve?")
for q in test_questions:
    result = needs_retrieval(q)
    print(f"  {'✅ Yes' if result else '⏩ Skip'} — '{q}'")

In [ ]:
# Check 2: Are the retrieved docs actually relevant to the question?
RELEVANCE_THRESHOLD = 0.35

def retrieve_with_relevance_check(question, threshold=RELEVANCE_THRESHOLD):
    q_emb = model.encode(question)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:2]

    top_score = scores[top_idx[0]]
    relevant_docs = [docs[i] for i in top_idx if scores[i] >= threshold]

    return relevant_docs, float(top_score)


# Good match
q1 = "How do I add items to a list?"
results, score = retrieve_with_relevance_check(q1)
print(f"Question: '{q1}'")
print(f"Top score: {score:.3f} — {'relevant ✅' if score >= RELEVANCE_THRESHOLD else 'not relevant ❌'}")
for r in results:
    print(f"  → {r}")

print()

# Bad match — question not covered in our docs
q2 = "What is the weather like today?"
results2, score2 = retrieve_with_relevance_check(q2)
print(f"Question: '{q2}'")
print(f"Top score: {score2:.3f} — {'relevant ✅' if score2 >= RELEVANCE_THRESHOLD else 'not relevant ❌'}")
print("  → Skip generation, tell user we don't have this info")

In [ ]:
# Check 3: Is the generated answer supported by the retrieved docs?

def is_answer_grounded(answer, retrieved_docs):
    """Check if the answer is semantically close to the retrieved docs."""
    if not retrieved_docs:
        return False
    ans_emb  = model.encode(answer)
    doc_embs = model.encode(retrieved_docs)
    scores   = cosine_similarity([ans_emb], doc_embs)[0]
    return float(np.max(scores)) >= 0.4  # At least one doc supports the answer


retrieved = ["Python lists are ordered and allow duplicate values.",
             "The append() method adds an item to the end of a list."]

good_answer = "Lists are ordered collections and you can add items with append()."
bad_answer  = "Lists in Python use hash tables internally for O(1) lookup."

print(f"Good answer grounded? {is_answer_grounded(good_answer, retrieved)} ✅")
print(f"Bad  answer grounded? {is_answer_grounded(bad_answer,  retrieved)} ❌")

In [ ]:
# Full Self-RAG pipeline

def self_rag(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Step 1: Do we need retrieval?
    if not needs_retrieval(question):
        print("[SKIP] No retrieval needed — answering directly")
        answer = llm(question, max_tokens=100)
        print(f"Answer: {answer}")
        return

    # Step 2: Retrieve and check relevance
    retrieved, top_score = retrieve_with_relevance_check(question)
    if not retrieved:
        print(f"[FAIL] Retrieved docs not relevant (score={top_score:.2f}) — cannot answer")
        return
    print(f"[OK]   Retrieved {len(retrieved)} relevant docs (score={top_score:.2f})")

    # Step 3: Generate with Claude
    context = "\n".join(retrieved)
    prompt = f"Answer using only this context:\n{context}\n\nQuestion: {question}\nAnswer:"
    answer = llm(prompt, max_tokens=150)

    # Step 4: Check if answer is grounded
    if is_answer_grounded(answer, retrieved):
        print(f"[OK]   Answer is grounded in sources")
        print(f"Answer: {answer}")
    else:
        print("[WARN] Answer not grounded — regenerating with stricter prompt")
        stricter = f"Answer in one sentence using ONLY this exact text:\n{context}\n\nQuestion: {question}"
        answer = llm(stricter, max_tokens=100)
        print(f"Answer: {answer}")


self_rag("How do Python lists work?")
print()
self_rag("Calculate 100 divided by 5")
print()
self_rag("What is the capital of France?")


## Key Takeaways 🎯

Self-RAG adds **three quality gates**:

1. **Do I need to retrieve?** → Skip for simple questions
2. **Are the retrieved docs relevant?** → Retry if not
3. **Is my answer grounded?** → Regenerate if not

Each gate catches a different failure mode. Together they make RAG much more reliable.

---
Next: `03_Multi_Hop_RAG.ipynb`